In [1]:
import json
import requests
import pandas as pd
from io import StringIO

In [2]:
hist_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["Diagnostic Slide"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["svs"]}}]}
expr_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["RNA-Seq"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["tsv"]}}]}
text_filter = {"op": "and", "content": [{"op": "in", "content": {"field": "cases.project.program.name", "value": ["TCGA"]}}, {"op": "in", "content": {"field": "files.data_type", "value": ["Pathology Report"]}}, {"op": "in", "content": {"field": "files.data_format", "value": ["pdf"]}}]}

In [3]:
def get_file_metadata(cohort_filter):
    response = requests.post(
        "https://api.gdc.cancer.gov/files",
        json={
            "filters": cohort_filter,
            "fields": ",".join(["file_name", "cases.project.project_id", "cases.submitter_id"]),
            "format": "TSV",
            "size": str(100_000),
        },
    )
    df = pd.read_csv(StringIO(response.text), sep="\t")
    df = df.rename(columns={"cases.0.project.project_id": "project_id", "cases.0.submitter_id": "submitter_id"}) # these files all map to a single case
    df["file_name"] = df["file_name"].str.replace(".svs", "").str.replace(".rna_seq.augmented_star_gene_counts.tsv", "").str.replace(".PDF", "")
    return df

hist_df = get_file_metadata(hist_filter)
expr_df = get_file_metadata(expr_filter)
text_df = get_file_metadata(text_filter)

In [4]:
project = "TCGA-COAD"
suffix = project.split("-")[1].lower()

for df, fname in [
    (expr_df, "expr"),
    (hist_df, "hist"),
    (text_df, "text"),
]:
    df = df[df["project_id"] == project]
    fids = set(df["file_name"])
    with open(f"../demos/with-all-data/manifests/{fname}.json", "r") as f:
        temp = json.load(f)
    filtered = [x for x in temp if x["file_id"] in fids]
    with open(f"../demos/manifests/{fname}-{suffix}.json", "w") as f:
        json.dump(filtered, f, indent=4)